# 강의 06 · 실습 7 — 패턴 1 병렬화 · (3) 변형

## 1. 문제상황

- 인사팀은 신입 사원 교육이 잡힐 때마다 교육 안내 메일을 보냅니다.
- 담당자는 일정·장소·대상이 대강 적힌 메모를 보고 일정 안내, 준비 자료 목록, 첫머리 인사말을 따로 써서 한 통의 메일로 합칩니다.
- 세 부분은 서로 참고할 것이 없는데도 담당자는 차례대로 씁니다. 합친 뒤에는 말투가 제각각이라 메일 전체를 다시 존댓말 안내문으로 고쳐 씁니다.
- 교육 회차가 늘어나면 담당자는 세 부분 작성과 마지막 고쳐 쓰기를 그만큼 반복해야 합니다.

## 2. 문제와 목표

- **문제**: 서로 독립인 세 가지 작성(일정 안내·준비 자료·인사말)을 사람이 차례대로 하고, 합친 결과를 다시 손으로 고쳐 씁니다. 네 가지 일이 교육 회차만큼 반복됩니다.
- **목표**: 교육 메모만 넣으면 세 노드가 동시에 각자의 부분을 쓰고, 결합 노드가 정해진 순서로 이어 붙이고, 다듬기 노드가 전체를 존댓말 안내 메일로 고쳐 쓰는 처리 흐름을 만듭니다.
    - 세 노드: agenda(일정과 장소 한 문장), prep(준비 자료 2개), greet(첫머리 인사말 한 문장).
    - 결합 노드: aggregator — 모델을 부르지 않고 인사말, 일정 안내, 빈 줄, 「준비할 자료:」, 준비 자료 순서로 이어 붙입니다.
    - 다듬기 노드: polish — 이어 붙인 본문을 내용을 바꾸지 않고 존댓말 안내 메일로 고쳐 씁니다(모델 1회).
- **목표 달성 여부의 판정 기준**: 교육 메모를 입력했을 때, 세 작성 노드가 같은 단계에서 진입하고, 결합 노드가 그 뒤에 한 번 돌고, 다듬기 노드가 결합 노드 뒤에 한 번 도는 순서를 실행 결과에서 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec06_ex07_s3_diagram.svg)

## 4. 단계별 요구사항

1. **상태를 정의합니다.**
    - 교육 메모(`memo`), 일정 안내(`agenda`), 준비 자료 목록(`prep`), 인사말(`greet`), 이어 붙인 본문(`body`), 완성 메일(`mail`) 키 여섯 개를 가지는 상태를 선언합니다.
    - 키 여섯 개 외의 값은 상태에 들어가지 않습니다.
2. **작성 노드 세 개를 만듭니다.**
    - agenda 노드는 메모의 일정과 장소를 한 문장으로 정리해 `agenda` 키에 씁니다.
    - prep 노드는 참석자가 미리 준비할 자료 2개를 뽑아 `prep` 키에 씁니다.
    - greet 노드는 신입 사원에게 보내는 첫머리 인사말을 한 문장으로 써서 `greet` 키에 씁니다.
    - 세 노드는 같은 메모를 입력으로 받되 시스템 프롬프트가 다르고, 서로 다른 키에만 씁니다.
3. **결합 노드를 만듭니다.**
    - aggregator 노드는 모델을 부르지 않고, 상태의 인사말·일정 안내·준비 자료를 정해진 순서(인사말, 일정 안내, 빈 줄, 준비할 자료, 준비 자료 목록)로 이어 붙여 `body` 키에 씁니다.
4. **다듬기 노드를 만듭니다.**
    - polish 노드는 상태의 `body`를 읽고, 모델을 불러 내용을 바꾸지 않은 채 존댓말 안내 메일 형식으로 고쳐 쓴 결과를 `mail` 키에 씁니다.
5. **그래프에 노드를 등록합니다.**
    - 다섯 노드를 이름과 함께 그래프에 등록합니다.
6. **엣지를 연결합니다.**
    - START에서 agenda·prep·greet 세 노드로 가는 고정 엣지를 각각 놓습니다.
    - 세 노드에서 aggregator로 가는 고정 엣지를 각각 놓습니다.
    - aggregator 뒤에는 polish를, polish 뒤에는 END를 고정 엣지로 연결합니다.
    - 조건부 엣지는 추가하지 않습니다.
7. **그래프를 컴파일하고 실행합니다.**
    - 교육 메모를 넣고, 노드가 하나 끝날 때마다 어느 노드가 상태의 어느 키를 채웠는지 화면에 출력한 뒤, 완성 메일을 출력합니다.
    - 값(`MEMO`, 교육 메모)은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
    - 각 노드 함수는 진입할 때 「[노드 이름] 진입」 줄을 출력합니다.

## 5. 코드 골격 — LangGraph 5단

랭그래프(LangGraph)로 그래프를 세우는 순서는 다음 다섯 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 다섯 단계와 하나씩 대응합니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 상태 정의 | 노드들이 함께 읽고 쓸 키를 선언합니다 | `class MailState(TypedDict)` | 1 |
| ② 노드 함수 정의 | 상태를 받아 바뀐 키만 돌려주는 함수를 만듭니다 | `def write_agenda(state) -> dict` | 2, 3, 4 |
| ③ 그래프 빌더 생성과 노드 등록 | 빈 그래프를 열고 함수에 이름을 붙여 등록합니다 | `StateGraph(MailState)`, `add_node` | 5 |
| ④ 엣지 연결 | START에서 나뉘고, 한 곳으로 모이고, 직렬로 이어지는 순서를 정합니다 | `add_edge` | 6 |
| ⑤ 컴파일과 실행 | 연결을 확정하고 입력을 넣어 실행합니다 | `compile()`, `stream()` | 7 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델을 준비합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
import os

from dotenv import load_dotenv, find_dotenv
from typing import TypedDict

from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
print("모델 준비를 마쳤습니다.")

# 주어진 자료: 교육 메모 MEMO — 값을 그대로 씁니다
MEMO = ("신입 교육 10월 13일(월) 10시~17시, 본사 3층 대강의실. "
        "대상은 10월 입사자 전원. 노트북 지참, 사원증은 당일 배부. "
        "점심 제공. 문의 인사팀 내선 1234.")

### 단계 ① — 상태 정의 (요구사항 1)

그래프가 도는 동안 모든 노드가 함께 읽고 쓰는 키를 선언합니다. `TypedDict`로 선언한 여섯 개의 키가 이 그래프에서 오가는 데이터의 전부입니다. 세 작성 노드가 각각 다른 키에 쓰도록 `agenda`·`prep`·`greet` 키 세 개를 따로 두고, 결합 결과와 다듬은 결과도 `body`·`mail` 키 두 개로 나눕니다.

In [ ]:
# 여기에 단계 ①(상태 정의)을 작성합니다.

### 단계 ② — 노드 함수 정의 (요구사항 2, 3, 4)

- 노드는 상태를 인자로 받아 딕셔너리를 돌려주는 파이썬 함수입니다. 돌려준 딕셔너리가 상태의 해당 키를 덮습니다.
- 세 작성 노드는 같은 메모를 `HumanMessage`로 받지만, `SystemMessage`가 서로 다르므로 서로 다른 결과를 만듭니다. 세 노드는 서로의 결과를 읽지 않으므로 동시에 돌 수 있습니다.
- aggregator 노드는 모델을 부르지 않습니다. 상태에 담긴 세 결과를 문자열로 이어 붙이기만 합니다.
- polish 노드는 aggregator가 만든 `body`를 읽으므로 aggregator 뒤에만 돌 수 있습니다. 모델을 한 번 더 부르는 직렬 단계입니다.

In [ ]:
# 여기에 단계 ②(작성 노드 세 개, 결합 노드, 다듬기 노드 정의)를 작성합니다.

### 단계 ③ — 그래프 빌더 생성과 노드 등록 (요구사항 5)

`StateGraph`에 상태를 넘겨 빈 그래프를 열고, `add_node`로 함수마다 이름을 붙여 등록합니다. 여기서 붙인 이름은 뒤의 엣지 연결에서 그대로 쓰입니다.

In [ ]:
# 여기에 단계 ③(그래프 빌더 생성과 노드 등록)을 작성합니다.

### 단계 ④ — 엣지 연결 (요구사항 6)

`add_edge`는 고정된 순서로 연결합니다. START에서 세 노드로 가는 엣지가 세 개이므로 세 노드는 같은 단계에서 동시에 시작합니다. 세 노드에서 aggregator로 가는 엣지가 세 개이므로 aggregator는 셋이 모두 끝난 뒤에 한 번 돕니다. aggregator에서 polish로 가는 엣지가 하나이므로 polish는 aggregator 뒤에 직렬로 돕니다.

In [ ]:
# 여기에 단계 ④(엣지 연결)를 작성합니다.

### 단계 ⑤ — 컴파일과 실행 (요구사항 7)

`compile()`이 연결을 확정해 실행 가능한 그래프를 돌려줍니다. `stream`은 노드가 하나 끝날 때마다 그 노드가 바꾼 키를 내보냅니다. 아래에서는 신입 사원 교육 메모를 넣습니다.

In [ ]:
# 여기에 단계 ⑤(컴파일과 실행)를 작성합니다.

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 세 가지를 확인합니다.

1. `agenda`·`prep`·`greet` 세 노드의 「진입」 줄이 `aggregator`의 진입 줄보다 먼저 출력됩니다. 세 노드끼리의 순서는 실행할 때마다 달라질 수 있습니다.
2. `aggregator`의 진입 줄 뒤에 `polish`의 진입 줄이 한 번 출력됩니다. polish는 aggregator가 채운 `body`를 읽으므로 그 뒤에만 돕니다.
3. 최종 상태의 키는 `agenda`·`body`·`greet`·`mail`·`memo`·`prep` 여섯 개이고, 완성 메일은 `body`의 내용을 유지한 채 존댓말 안내문으로 바뀌어 있습니다.

세 가지가 모두 확인되면 완성입니다. 하나라도 다르면 해당 단계의 코드를 다시 봅니다. polish가 aggregator보다 먼저 출력되거나 `body`가 비어 있다는 오류가 나면 단계 ④의 엣지 방향을 다시 봅니다.